# 03. FranceCrops development parcels

[Notebook 01](01_external_reference_audit.ipynb) introduces the external sources and their roles. FranceCrops contributes cabbage and table potato to development. The current batch request targets cabbage. Earlier plans can still contain potato.

Saved outputs may describe an earlier run. [Notebook 08](08_dataset_summary.ipynb) shows the parcels used in the saved experiment.

## 1. Settings for this run

Review the settings before running the notebook from top to bottom.

| Setting | What it controls |
|---|---|
| `reference_mode` | Controls local reference reuse or downloading missing files. Offline mode also disables satellite access. |
| `run_satellite_collection` | Enables Earth Engine submission and retrieval, subject to the notebook's reference mode. |
| `submit_only` | Queues cloud work for retrieval in a later run. |
| `candidate_parcels_per_crop` | Maximum candidate parcels per crop in each newly planned batch. |
| `extra_batches` | Highest numbered extra slot to consider in the current round, after the regular batch. |
| `download_workers` | Controls local transport concurrency. |
| `intervals_per_request` | Controls the number of half-month intervals in a retrieval request. |

A **campaign** holds the collection goal and its records, a **round** groups batch plans, and a **batch** is a set of selected parcels with an imagery plan. Rerunning with the same extra-slot limit revisits the current round and can resume unfinished work. For example, changing 25 to 26 makes slot 0026 available in that round if eligible parcels remain. [Collection guide](README.md)

In [ ]:
# Edit these values before running collection.
reference_mode = "reuse"  # reuse | download_missing | offline
run_satellite_collection = True  # offline always disables satellite calls
submit_only = False  # Queue exports. Set False to collect finished results.
candidate_parcels_per_crop = 100
extra_batches = 0  # One separately named top-up batch.
top_up_number = 2  # Resume the second top-up batch.
download_workers = 2
intervals_per_request = 6


In [ ]:
from pathlib import Path
import os
import sys
repository_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), Path.cwd())
source_folder = str(repository_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)
from crop_pipeline import notebook_setup, source_workflow
from crop_pipeline.notebook_display import show_table, show_mapping, show_items
reference_options = source_workflow.reference_options(reference_mode)
notebook_setup.prepare_environment(repository_root, install_missing=reference_mode != "offline")
os.chdir(repository_root)

from crop_pipeline.sources import adaptive, annual
from crop_pipeline import progress as pipeline_progress
from crop_pipeline.windows.seasons import file_sha256
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./data/pipeline_work/inventory")).expanduser().resolve()
annual_archive_root = annual.archive_root(local_settings, report_folder, repository_root=repository_root)
run_name = adaptive.COLLECTION_NAME
run_satellite_collection = run_satellite_collection and reference_mode != "offline"
authenticate_now = run_satellite_collection

from crop_pipeline.sources import francecrops as francecrops_workflow
source_name = "francecrops"
inward_buffer_metres = 10  # Polygon interiors, preserving the existing rule.
revision = francecrops_workflow.DEFAULT_REVISION
source_table_folder = report_folder.parent / "francecrops/source_metadata" / revision

# Resolve account configuration before source transfers; authentication occurs during collection.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)
print(f"Source: {source_name} | References: {reference_mode} | Round limit: {candidate_parcels_per_crop} per crop")
print(f"Inward buffer: {inward_buffer_metres} m | Satellite collection: {run_satellite_collection}")
print("Archive:", annual_archive_root)
print("Backend: Earth Engine batch assets → Parquet transport → annual TIFFs")
notebook_progress = pipeline_progress.start_notebook(
    '02_francecrops_preparation', source_workflow.STAGES, report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "source": source_name},
)

## 2. Obtain references

Load the pinned reference tables. The source reader links the declared crop code to parcel geometry by parcel ID.

In [ ]:
pipeline_progress.begin_stage("Obtain references")
table_paths = francecrops_workflow.download_reference_tables(
    source_table_folder, revision=revision,
    download_if_missing=reference_options["download_if_missing"],
)
reference_note = f"Pinned FranceCrops release {revision}; cabbage/table potato only"
show_mapping(table_paths, title="Reference files", key_label="Reference", value_label="File")
pipeline_progress.end_stage()


## 3. Check parcel eligibility

Check usable reference geometry and the CHU cabbage and PTC table-potato labels. The planner samples polygon interiors using the existing 10 m inward buffer.

In [ ]:
pipeline_progress.begin_stage("Validate references")
reference_features, reference_audit = francecrops_workflow.load_target_parcels(table_paths, revision, return_audit=True)
show_mapping(reference_audit, title="Reference audit", key_label="Check", value_label="Result")
print("Edge handling: 10 m inward buffer on parcel polygons.")
pipeline_progress.end_stage()


## 4. Plan or resume batches

The planner uses eligible field footprints to create imagery requests. Saved plans carry their original selections and project splits. The batch table shows selected parcels and planned tiles for plans returned in this run.

In [ ]:
pipeline_progress.begin_stage("Prepare collection round")
adaptive_batches = adaptive.prepare_round_batches(
    "francecrops", reference_features, annual_archive_root, report_folder,
    provenance={"dataset_revision": revision, "tables": {name: file_sha256(path) for name, path in table_paths.items()},
                "unavailable_target_classes": ["carrot"], "upstream_splits_preserved": True, "reference_audit": reference_audit},
    inward_buffer_metres=inward_buffer_metres, parcels_per_crop=candidate_parcels_per_crop, extra_batches=extra_batches,
    candidate_crops=("cabbage",),
    top_up_number=top_up_number,
    development_top_up={"cabbage": ("train", "validation")},
)
annual_plan, adaptive_context = next(((plan, context) for plan, context in adaptive_batches if plan is not None), (None, adaptive_batches[-1][1]))
print("Active/resumed batches:", len(adaptive_batches))
print("Selected parcels:", sum(context["candidate_parcels"] for plan, context in adaptive_batches if plan is not None))
print("Tiles:", sum(plan["estimate"]["tiles"] for plan, context in adaptive_batches if plan is not None))
show_items(sorted({year for plan, context in adaptive_batches if plan is not None for year in plan["years"]}), title="Years")
pipeline_progress.set_workload({"folder": str(annual_archive_root), "source": source_name,
    "plans": [plan["revision"] for plan, context in adaptive_batches if plan is not None],
    "satellite_collection": run_satellite_collection})
pipeline_progress.end_stage()

# Count parcels and imagery tiles separately.
show_table([
    {"Batch": Path(context["folder"]).name, "Selected parcels": context["candidate_parcels"],
      "Planned tiles": plan["estimate"]["tiles"]}
    for plan, context in adaptive_batches if plan is not None
], title="Parcel and imagery workload by batch")

## 5. Collect annual imagery

Earth Engine uses the planned field footprints and years to find and process Sentinel-2 imagery. With `submit_only = True`, it can continue processing after submission finishes and the laptop is closed. On a later run, `submit_only = False` retrieves completed results, caches selected-pixel tables as Parquet and assembles annual TIFFs. Matching verified work can be reused. [Transport and caching](../../docs/reference/transport.md)

In [ ]:
pipeline_progress.begin_stage("Collect annual imagery")
collection_result = adaptive.collect_batches(
    adaptive_batches, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
    submit_only=submit_only,
    download_workers=download_workers, intervals_per_request=intervals_per_request,
)
pipeline_progress.end_stage("skipped" if not run_satellite_collection else "completed" if collection_result["status"] == "completed" else "pending")


## 6. Review collection status

Use the batch status to see which work finished and which can resume later. Verify the parcels and windows that pass imagery checks through preprocessing.

In [ ]:
pipeline_progress.begin_stage("Summary and next action")
# Status comes from this run's collection response.
show_mapping({"Collection status": collection_result["status"].replace("_", " ").title()},
             title="Collection result")
show_table([
    {"Batch": entry["batch"], "Status": entry["result"]["status"].replace("_", " ").title()}
    for entry in collection_result.get("batches", [])
], title="Collection status by batch")
source_workflow.print_summary(
    source_name, adaptive_batches, collection_result, reference_mode,
    inward_buffer_metres, reference_note=reference_note,
)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()

## Saved outputs

Browse reference files, batch states and imagery through the organized collection index.

In [ ]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(repository_root)
print("Organized collection outputs:", repository_root / "pipeline_outputs/collection")
print("Next: data_collection/08_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.")


Continue to [Notebook 08](08_dataset_summary.ipynb) for the combined dataset profile and [Preprocessing 01](../preprocessing/01_imagery_quality.ipynb) for imagery quality and coverage.